In [9]:
%load_ext autoreload
%autoreload 2

from dataclasses import dataclass
from sklearn.metrics import accuracy_score, roc_auc_score
import torch
from torch import nn
from torch.nn import CrossEntropyLoss

from tfmplayground.model import NanoTabPFNModel
from tfmplayground.priors import PriorDumpDataLoader
from tfmplayground.train import train
from tfmplayground.utils import get_default_device
from tfmplayground.interface import NanoTabPFNClassifier
from tfmplayground.callbacks import ConsoleLoggerCallback, WandbLoggerCallback

from tfmplayground.evaluation import get_openml_predictions, TOY_TASKS_CLASSIFICATION, TABARENA_TASKS

from gtfm.viz.imshow import imshow

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
# @dataclass
# class ModelConfig:
#     num_attention_heads: int = 6
#     embedding_size: int = 192
#     mlp_hidden_size: int = 768
#     num_layers: int = 6
#     num_outputs: int = 10
#     mask_attn: bool = True

@dataclass
class ModelConfigSmall:
    num_attention_heads: int = 4
    embedding_size: int = 64
    mlp_hidden_size: int = 128
    num_layers: int = 3
    num_outputs: int = 10
    mask_attn: bool = True


@dataclass
class PriorConfig:
    filename: str = '../data/tabicl_4k_50x3.h5'
    num_steps: int = 50
    batch_size: int = 50

@dataclass
class TrainConfig:
    epochs: int = 5
    accumulate: int = 1
    lr: float = 1e-4
    multigpu: bool = False
    runname: str = 'nanotabpfn'
    workdir: str = '../logs/'

# model_cfg = ModelConfig()
model_cfg = ModelConfigSmall()
prior_cfg = PriorConfig()
train_cfg = TrainConfig()
device = get_default_device()

In [11]:
model = NanoTabPFNModel(
    **model_cfg.__dict__
)
criterion = CrossEntropyLoss()

In [12]:
prior = PriorDumpDataLoader(**prior_cfg.__dict__, device=device)

In [13]:
class ToyEvaluationLoggerCallback(ConsoleLoggerCallback):
    def __init__(self, tasks):
        self.tasks = tasks

    def on_epoch_end(self, epoch: int, epoch_time: float, loss: float, model, **kwargs):
        classifier = NanoTabPFNClassifier(model, device)
        predictions = get_openml_predictions(model=classifier, tasks=self.tasks)
        scores = []
        for dataset_name, (y_true, y_pred, y_proba) in predictions.items():
            scores.append(accuracy_score(y_true, y_pred))
        avg_score = sum(scores) / len(scores)
        print(f'epoch {epoch:5d} | time {epoch_time:5.2f}s | mean loss {loss:5.2f} | avg accuracy {avg_score:.3f}',
              flush=True)
class ProductionEvaluationLoggerCallback(WandbLoggerCallback):
    def __init__(self, project: str, name: str = None, config: dict = None, log_dir: str = None):
        super().__init__(project, name, config, log_dir)

    def on_epoch_end(self, epoch: int, epoch_time: float, loss: float, model, **kwargs):
        classifier = NanoTabPFNClassifier(model, device)
        print('openml evaluation...', flush=True)
        predictions = get_openml_predictions(
            max_n_samples = 1_000,
            max_n_features = 100,
            model=classifier, classification=True, tasks=TABARENA_TASKS
        )
        print('evaluation done.', flush=True)
        scores = []
        for dataset_name, (y_true, y_pred, y_proba) in predictions.items():
            scores.append(roc_auc_score(y_true, y_proba, multi_class='ovr'))
        avg_score = sum(scores) / len(scores)
        self.wandb.log({
            'epoch': epoch,
            'epoch_time': epoch_time,
            'mean_loss': loss,
            'tabarena_avg_roc_auc': avg_score
        })
        print(f'epoch {epoch:5d} | time {epoch_time:5.2f}s | mean loss {loss:5.2f} | avg roc auc {avg_score:.3f}',
              flush=True)

# callbacks = [ProductionEvaluationLoggerCallback('tfm', train_cfg.runname)]
callbacks = [ProductionEvaluationLoggerCallback('tfm')]
train_cfg.runname=callbacks[0].wandb.run.name
# callbacks = [ToyEvaluationLoggerCallback(TOY_TASKS_CLASSIFICATION)]

In [14]:
# model.to(device)

# classification_task = isinstance(criterion, nn.CrossEntropyLoss)
# regression_task = not classification_task

# for i, full_data in enumerate(prior):
#     single_eval_pos = full_data['single_eval_pos']
#     data = (full_data['x'].to(device),
#             full_data['y'][:, :single_eval_pos].to(device),
#             full_data['adj'].to(device))
#     if (torch.isnan(data[0]).any() or torch.isnan(data[1]).any()):
#         continue
#     targets = full_data['target_y'].to(device)

#     if regression_task:
#         y_mean = data[1].mean(dim=1, keepdim=True)
#         y_std = data[1].std(dim=1, keepdim=True) + 1e-8
#         y_norm = (data[1] - y_mean) / y_std
#         data = (data[0], y_norm, data[2])

#     output = model(data, single_eval_pos=single_eval_pos)

#     break

In [15]:
# trained_model, loss = train(
ret = train(
    model=model,
    prior=prior,
    criterion=criterion,
    epochs=train_cfg.epochs,
    accumulate_gradients=train_cfg.accumulate,
    lr=train_cfg.lr,
    device=device,
    callbacks=callbacks,
    ckpt=None,
    multi_gpu=train_cfg.multigpu,
    run_name=train_cfg.runname,
    workdir=train_cfg.workdir,
)

openml evaluation...


INFO:openml.datasets.dataset:pickle write anneal
INFO:openml.datasets.dataset:pickle write blood-transfusion-service-center
INFO:openml.datasets.dataset:pickle write credit-g
INFO:openml.datasets.dataset:pickle write diabetes


evaluation done.
epoch     1 | time  8.96s | mean loss  1.84 | avg roc auc 0.516
openml evaluation...


INFO:openml.datasets.dataset:pickle write anneal
INFO:openml.datasets.dataset:pickle write blood-transfusion-service-center
INFO:openml.datasets.dataset:pickle write credit-g
INFO:openml.datasets.dataset:pickle write diabetes


evaluation done.
epoch     2 | time  8.62s | mean loss  1.30 | avg roc auc 0.505
openml evaluation...


...

evaluation done.
epoch     4 | time  8.66s | mean loss  0.90 | avg roc auc 0.410
openml evaluation...


INFO:openml.datasets.dataset:pickle write anneal
INFO:openml.datasets.dataset:pickle write blood-transfusion-service-center
INFO:openml.datasets.dataset:pickle write credit-g
INFO:openml.datasets.dataset:pickle write diabetes


evaluation done.
epoch     5 | time  8.55s | mean loss  0.80 | avg roc auc 0.414


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


epoch,▁▃▅▆█
epoch_time,█▄▁▄▃
mean_loss,█▄▃▂▁
tabarena_avg_roc_auc,█▇▁▁▁
epoch,5
epoch_time,8.55361
mean_loss,0.79561
tabarena_avg_roc_auc,0.41378


In [16]:

# imshow(adjs, n_rows=10)